[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-04-branching-merging.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Branching and Merging Flows
**certified-journeys / metaflow-certified** · Metaflow for ML Engineers

> **Goal for today:** Implement static branches, dynamic foreach branches, and join steps that collect results from parallel tasks — and verify the artifacts locally with the Client API.


In [ ]:
%pip install -q metaflow


## Step 1 · Static Branching — two named parallel steps

Metaflow lets you **split a flow into named parallel branches** by listing multiple step names in `self.next()`. Each branch runs independently (possibly on different machines), and a join step reunites them.

```
start → branch_a ↘
              join → end
        branch_b ↗
```

Rules for static branching:
- Call `self.next(self.branch_a, self.branch_b)` to fan out.
- The join step must list **all** branch step names as `inputs`: `@step def join(self, inputs): …`.
- Artifacts set inside a branch are scoped to that branch task — use `inputs` to collect them.


In [ ]:
%%writefile static_branch_flow.py
from metaflow import FlowSpec, step

class StaticBranchFlow(FlowSpec):
    """
    Demonstrates a two-way static branch:
      start → branch_a  ↘
                          join → end
              branch_b  ↗
    """

    @step
    def start(self):
        self.shared_input = "hello from start"
        # Fan out into two named steps simultaneously
        self.next(self.branch_a, self.branch_b)

    @step
    def branch_a(self):
        # Each branch sees artifacts from the step before the split
        self.result = f"Branch A processed: {self.shared_input.upper()}"
        self.next(self.join)

    @step
    def branch_b(self):
        self.result = f"Branch B processed: {self.shared_input[::-1]}"
        self.next(self.join)

    @step
    def join(self, inputs):
        # `inputs` is a list of task objects — one per branch
        self.results = [inp.result for inp in inputs]
        # merge() promotes a common artifact from all branches into this task
        self.merge_artifacts(inputs, exclude=["result"])
        self.next(self.end)

    @step
    def end(self):
        print("All branch results:", self.results)

if __name__ == "__main__":
    StaticBranchFlow()


In [ ]:
!python static_branch_flow.py run


### What just happened?

- **`self.next(self.branch_a, self.branch_b)`** tells Metaflow to schedule both steps in parallel — locally they run sequentially, but on a remote executor they can run on separate machines.
- **`inputs`** in the join step is an iterable of `Task` objects. Each `inp.result` retrieves the artifact from that branch.
- **`merge_artifacts`** promotes artifacts that are identical across all branches (like `shared_input`) so you can access them without iterating `inputs`.
- The DAG is fully declared at class-definition time — Metaflow validates it before any step runs.


## Step 2 · Dynamic `foreach` Branching — fan out over a list

`foreach` is the dynamic counterpart to static branching. Instead of naming every branch, you pass a list and Metaflow spawns **one task per element**.

| Feature | Static branch | foreach branch |
|---------|---------------|----------------|
| Number of branches | Fixed at code time | Dynamic — driven by data |
| Branch names | Explicit step names | Single step, `self.input` holds the element |
| Use case | Distinct logic per branch | Same logic over many items |

Inside the foreach step, **`self.input`** (singular) holds the current element. The join step still receives **`inputs`** (plural).


In [ ]:
%%writefile foreach_flow.py
from metaflow import FlowSpec, step

class ForeachFlow(FlowSpec):
    """
    Demonstrates a dynamic foreach branch:
      start → process (×N, one per model) → gather → end
    """

    @step
    def start(self):
        # Define the list to iterate over — one task per element
        self.models = ["linear", "tree", "forest", "xgboost"]
        self.next(self.train_model, foreach="models")

    @step
    def train_model(self):
        # self.input holds the current element (e.g. "linear")
        import random, time
        random.seed(hash(self.input))
        # Simulate training: record model name and a fake accuracy score
        self.model_name = self.input
        self.accuracy = round(random.uniform(0.70, 0.99), 4)
        print(f"  Trained {self.model_name}: accuracy={self.accuracy}")
        self.next(self.gather)

    @step
    def gather(self, inputs):
        # Collect results from every foreach task
        self.scores = {
            inp.model_name: inp.accuracy
            for inp in inputs
        }
        self.best_model = max(self.scores, key=self.scores.get)
        self.next(self.end)

    @step
    def end(self):
        print("\nAll scores:", self.scores)
        print("Best model:", self.best_model, "→", self.scores[self.best_model])

if __name__ == "__main__":
    ForeachFlow()


In [ ]:
!python foreach_flow.py run


### What just happened?

- **`self.next(self.train_model, foreach="models")`** spawns one `train_model` task for each element in `self.models`.
- **`self.input`** inside the foreach step is the *current element* — here one of `"linear"`, `"tree"`, etc.
- The `gather` join step collects all four tasks via `inputs` and builds a dict of scores.
- Metaflow automatically parallelises foreach tasks when running on a remote executor (AWS Batch, Kubernetes), so 100 models is as easy as 4.


## Step 3 · The Join Step — collecting results via `inputs`

The join step is where both branching patterns converge. Understanding how `inputs` works is essential:

| `inputs` attribute | Meaning |
|--------------------|---------|
| `inputs[i].<attr>` | Artifact from branch i |
| `len(inputs)` | Number of parallel tasks that merged here |
| `self.merge_artifacts(inputs)` | Promote artifacts identical across all branches |
| `self.merge_artifacts(inputs, exclude=["x"])` | Promote everything except `x` |

After `merge_artifacts`, the promoted artifacts are accessible directly as `self.<attr>` in downstream steps — no need to re-iterate `inputs`.


In [ ]:
%%writefile join_demo_flow.py
from metaflow import FlowSpec, step

class JoinDemoFlow(FlowSpec):
    """
    Demonstrates merge_artifacts and explicit iteration in a join step.
    """

    @step
    def start(self):
        # common_config will be the same in every branch — good candidate for merge_artifacts
        self.common_config = {"dataset": "iris", "seed": 42}
        self.next(self.preprocess, self.feature_eng)

    @step
    def preprocess(self):
        self.step_name = "preprocess"
        self.artifacts_produced = ["cleaned_data", "train_idx", "test_idx"]
        self.next(self.combine)

    @step
    def feature_eng(self):
        self.step_name = "feature_eng"
        self.artifacts_produced = ["feature_matrix", "feature_names"]
        self.next(self.combine)

    @step
    def combine(self, inputs):
        # Iterate inputs to collect per-branch artifacts
        pipeline_stages = {}
        for inp in inputs:
            pipeline_stages[inp.step_name] = inp.artifacts_produced
        self.pipeline_stages = pipeline_stages

        # Promote common_config (identical in both branches) into self
        self.merge_artifacts(inputs, exclude=["step_name", "artifacts_produced"])
        self.next(self.end)

    @step
    def end(self):
        # common_config is now accessible directly thanks to merge_artifacts
        print("Config used:", self.common_config)
        print("Pipeline stages:")
        for stage, arts in self.pipeline_stages.items():
            print(f"  {stage}: {arts}")

if __name__ == "__main__":
    JoinDemoFlow()


In [ ]:
!python join_demo_flow.py run


### What just happened?

- **`self.merge_artifacts(inputs, exclude=[...])`** is the recommended way to handle artifacts that are unchanged across branches — it avoids duplicating the value and makes downstream steps cleaner.
- **Branch-specific artifacts** (like `step_name` and `artifacts_produced`) must be collected manually by iterating `inputs`.
- After `combine`, `self.common_config` is accessible in `end` as a regular attribute — no reference to `inputs` needed.
- If you forget to handle a conflicting artifact (different value in different branches) without `merge_artifacts`, Metaflow will raise a clear error.


## Step 4 · Inspecting Results with the Metaflow Client API

Metaflow stores every run's artifacts in a local metadata store (`.metaflow/` by default). The **Client API** lets you load any past run and inspect its artifacts interactively.

Key classes:
- `Flow('FlowName')` — access all runs of a flow
- `Run('FlowName/run_id')` — a specific run
- `Step('FlowName/run_id/step_name')` — all tasks of a step
- `Task('FlowName/run_id/step_name/task_id')` — a single task
- `task.data.<attr>` — retrieve a stored artifact


In [ ]:
from metaflow import Flow, Run

# --- Inspect the ForeachFlow run ---
flow = Flow("ForeachFlow")
latest = flow.latest_run
print(f"Latest ForeachFlow run: {latest.id}  finished={latest.finished}")

# List every step in the run
for step in latest:
    print(f"  step: {step.id}")


In [ ]:
# Drill into the foreach step — one task per model
train_step = latest["train_model"]
print("Tasks in 'train_model' step:")
for task in train_step:
    print(f"  task {task.id}: model={task.data.model_name}  accuracy={task.data.accuracy}")

# Access the join step artifacts
gather_task = next(iter(latest["gather"]))
print("\nGathered scores:", gather_task.data.scores)
print("Best model:", gather_task.data.best_model)


### What just happened?

- **`Flow('ForeachFlow').latest_run`** retrieves the most recent run without needing to remember a run ID — perfect for interactive exploration.
- **Iterating over a `Step`** yields all its tasks — in a foreach step that's one task per list element.
- **`task.data.<attr>`** deserialises the artifact from disk. Metaflow uses pickle by default but supports custom serialisers.
- This same Client API works identically whether the flow ran locally or on AWS Batch — the metadata store abstracts the location.


## Step 5 · Combining Both Branching Patterns

Real ML pipelines often use **static branches for distinct logic** (e.g. preprocessing vs. validation) and **foreach for hyperparameter sweeps or cross-validation folds**. They can be nested or sequenced.


In [ ]:
%%writefile combined_flow.py
from metaflow import FlowSpec, step

class CombinedBranchFlow(FlowSpec):
    """
    Static branch to prepare two datasets, then foreach to evaluate
    multiple models against the combined data.
    """

    @step
    def start(self):
        self.next(self.load_train, self.load_holdout)

    @step
    def load_train(self):
        # Simulate loading a training split
        self.split = "train"
        self.size = 800
        self.next(self.merge_splits)

    @step
    def load_holdout(self):
        # Simulate loading a holdout split
        self.split = "holdout"
        self.size = 200
        self.next(self.merge_splits)

    @step
    def merge_splits(self, inputs):
        # Collect sizes from both branches
        self.split_sizes = {inp.split: inp.size for inp in inputs}
        self.total_size = sum(self.split_sizes.values())
        # Now fan out over multiple models using foreach
        self.model_list = ["logreg", "svm", "rf"]
        self.next(self.evaluate, foreach="model_list")

    @step
    def evaluate(self):
        import random
        random.seed(hash(self.input) + self.total_size)
        self.model = self.input
        self.score = round(random.uniform(0.75, 0.98), 4)
        print(f"  {self.model} on {self.total_size} samples → {self.score}")
        self.next(self.report)

    @step
    def report(self, inputs):
        self.leaderboard = sorted(
            [(inp.model, inp.score) for inp in inputs],
            key=lambda x: -x[1]
        )
        self.next(self.end)

    @step
    def end(self):
        print("\nLeaderboard:")
        for rank, (model, score) in enumerate(self.leaderboard, 1):
            print(f"  #{rank} {model}: {score}")

if __name__ == "__main__":
    CombinedBranchFlow()


In [ ]:
!python combined_flow.py run


### What just happened?

- A **static branch** (train/holdout) and a **foreach branch** (models) are composed in the same flow — Metaflow handles the DAG correctly.
- `merge_splits` acts as both a join step (collecting from static branches) *and* a fan-out step (foreach over models). This is valid — a step can join and then immediately fork.
- **`self.total_size`** set in `merge_splits` is automatically available inside `evaluate` via `self.total_size` because Metaflow propagates parent artifacts to foreach tasks.
- The final leaderboard is built in a single join step — clean and reproducible.


In [ ]:
# Challenge: Extend ForeachFlow to also compute the standard deviation of all accuracy scores
# in the gather step, and print it alongside the best model.
#
# Hint: collect all scores as a list, then use Python's statistics.stdev()
#
# Your solution — add the std computation inside the gather step:
#
# import statistics
# self.score_list = [inp.accuracy for inp in inputs]
# self.score_std  = ???
#
# Then print it in end().
pass


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| Static branch | `self.next(self.a, self.b)` — fixed parallel steps with distinct logic |
| foreach branch | `self.next(self.step, foreach="list_attr")` — dynamic fan-out over data |
| `self.input` | Current element inside a foreach step (singular) |
| `inputs` param | List of task objects in a join step (plural) |
| `merge_artifacts` | Promotes identical artifacts from all branches; excludes conflicting ones |
| Client API | `Flow / Run / Step / Task` — inspect any past run's artifacts interactively |

> **Tip:** In a join step, `inputs` is a list of task objects — iterate over it to collect results from all parallel branches.

---
## What's next
**Day 5** → Scale individual steps to AWS Batch or Kubernetes with `@resources` and `--with batch`, and learn how to pass environment variables to remote steps.

Mark Day 4 complete in your [tracker](../index.html).
